# E22 MovieLens 电影推荐：用 NDCG@10 评估 Top‑K 排序

> 公开范例对照：[Recommenders Quick Start](https://github.com/recommenders-team/recommenders/blob/main/examples/00_quick_start/README.md)  
> 覆盖矩阵：推荐 / 排序指标（NDCG）

## 你将学到
1. 时间切分（按用户历史）避免泄漏。
2. **热门榜基线** vs 简易用户-物品相似度 / 共现。
3. 同一候选集上算 **NDCG@10**（及 Recall@K 辅助）。
4. 数据：优先本地/网络 MovieLens 100K 子集；失败则用合成交互（保证 evidence）。


In [ ]:
QUICK = True
RANDOM_STATE = 42
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 加载 MovieLens 或 DEMO 交互

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from wanglaoshi import DataAnalyzer

rng = np.random.default_rng(RANDOM_STATE)
DEMO = True
ratings = None
# 尝试本地
for cand in [Path('../../data/ml-100k/u.data'), Path('data/ml-100k/u.data')]:
    if cand.exists():
        ratings = pd.read_csv(cand, sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
        DEMO = False
        print('local', cand, ratings.shape)
        break
if ratings is None:
    try:
        from urllib.request import urlopen
        import io
        url = 'https://files.grouplens.org/datasets/movielens/ml-100k/u.data'
        raw = urlopen(url, timeout=8).read()
        ratings = pd.read_csv(io.BytesIO(raw), sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
        DEMO = False
        print('downloaded ml-100k', ratings.shape)
    except Exception as e:
        print('fallback DEMO', type(e).__name__, e)
        n_users, n_items, n = 80, 120, 2500
        ratings = pd.DataFrame({
            'user_id': rng.integers(1, n_users + 1, n),
            'item_id': rng.integers(1, n_items + 1, n),
            'rating': rng.integers(1, 6, n),
            'timestamp': np.sort(rng.integers(1_000_000_000, 1_100_000_000, n)),
        }).drop_duplicates(['user_id', 'item_id'])
        DEMO = True
# QUICK 子样
if QUICK and len(ratings) > 8000:
    keep_u = ratings['user_id'].drop_duplicates().sample(min(200, ratings.user_id.nunique()), random_state=RANDOM_STATE)
    ratings = ratings[ratings.user_id.isin(keep_u)].copy()
print('DEMO', DEMO, ratings.shape, DataAnalyzer(ratings).get_basic_info().get('shape'))


## 2. 按用户时间切分：最后一次交互进测试

In [ ]:
ratings = ratings.sort_values(['user_id', 'timestamp'])
# 每用户至少 2 条
vc = ratings.groupby('user_id').size()
ratings = ratings[ratings.user_id.isin(vc[vc >= 2].index)].copy()

def leave_one_last(g):
    g = g.sort_values('timestamp')
    return g.iloc[:-1], g.iloc[-1:]

train_parts, test_parts = [], []
for _, g in ratings.groupby('user_id'):
    tr, te = leave_one_last(g)
    train_parts.append(tr)
    test_parts.append(te)
train = pd.concat(train_parts)
test = pd.concat(test_parts)
print(train.shape, test.shape, 'users', test.user_id.nunique())


## 3. 热门榜基线 + 共现推荐

In [ ]:
# 热门：训练集全局频次
pop = train.groupby('item_id').size().sort_values(ascending=False)
# 用户已看集合
seen = train.groupby('user_id')['item_id'].apply(set).to_dict()

def recommend_popular(uid, k=10):
    s = seen.get(uid, set())
    rec = [i for i in pop.index if i not in s][:k]
    return rec

# 简易物品共现：对测试用户，取其训练物品的共现邻居投票
from collections import defaultdict, Counter
# 构建稀疏共现（QUICK：限制用户数）
user_items = train.groupby('user_id')['item_id'].apply(list).to_dict()
co = defaultdict(Counter)
for items in list(user_items.values())[: min(500, len(user_items))]:
    uniq = list(set(items))
    for a in uniq:
        for b in uniq:
            if a != b:
                co[a][b] += 1

def recommend_cooc(uid, k=10):
    s = seen.get(uid, set())
    scores = Counter()
    for it in s:
        scores.update(co.get(it, {}))
    ranked = [i for i, _ in scores.most_common(k + len(s)) if i not in s]
    if len(ranked) < k:
        # 回退热门补齐
        ranked += [i for i in recommend_popular(uid, k=k + 20) if i not in ranked and i not in s]
    return ranked[:k]


## 4. NDCG@10（同一候选长度）

In [ ]:
def dcg_at_k(rels, k=10):
    rels = np.asarray(rels[:k], dtype=float)
    if rels.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, rels.size + 2))
    return float((rels * discounts).sum())

def ndcg_at_k(recommended, relevant_set, k=10):
    rels = [1.0 if i in relevant_set else 0.0 for i in recommended[:k]]
    dcg = dcg_at_k(rels, k)
    ideal = dcg_at_k([1.0] * min(k, len(relevant_set)), k)
    return 0.0 if ideal == 0 else dcg / ideal

def eval_method(fn, k=10):
    scores = []
    for row in test.itertuples(index=False):
        uid, iid = row.user_id, row.item_id
        rec = fn(uid, k=k)
        scores.append(ndcg_at_k(rec, {iid}, k=k))
    return float(np.mean(scores))

ndcg_pop = eval_method(recommend_popular, 10)
ndcg_co = eval_method(recommend_cooc, 10)
print(dict(ndcg10_popular=ndcg_pop, ndcg10_cooc=ndcg_co, n_test_users=int(test.user_id.nunique()), DEMO=DEMO))
out = Path('outputs/22_movielens_ndcg')
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{
    'ndcg10_popular': ndcg_pop,
    'ndcg10_cooc': ndcg_co,
    'demo': DEMO,
}]).to_csv(out / 'ndcg_summary.csv', index=False)


## 附录
- Recommenders SAR 等可作进阶；本课强调 **指标与切分契约**，不绑重依赖。
- 作业：改成 NDCG@5；对负采样候选集固定后再比模型。
- 包优化：可选 `RankingMetrics.ndcg_at_k`。
